In [ ]:
import cv2
import numpy as np

import json

from pathlib import Path

import math

In [ ]:
# ======================================================
# Load State
# ======================================================


ROOT = Path.cwd().parent


STATE_FILE = (
    ROOT /
    "config" /
    "project_state.json"
)


with open(
    STATE_FILE,
    encoding="utf-8"
) as f:

    PROJECT_STATE=json.load(f)



PROJECT_STATE

In [ ]:
# ======================================================
# Load Edge
# ======================================================


EDGE_PATH = Path(

    PROJECT_STATE[
        "edge_image"
    ]

)



edges=cv2.imread(

    str(EDGE_PATH),

    cv2.IMREAD_GRAYSCALE

)


print(
edges.shape
)

In [ ]:
import matplotlib.pyplot as plt


plt.figure(
    figsize=(10,10)
)

plt.imshow(
    edges,
    cmap="gray"
)

plt.axis("off")

plt.title(
    "Edge Map"
)

plt.show()

In [ ]:
# ======================================================
# Detect Lines
# ======================================================


lines=cv2.HoughLinesP(

    edges,

    1,

    np.pi/180,

    threshold=80,

    minLineLength=50,

    maxLineGap=10

)



print(

"Lines:",

0 if lines is None else len(lines)

)

In [ ]:
# ======================================================
# Line Objects
# ======================================================


LINE_OBJECTS=[]


if lines is not None:


    for line in lines:


        x1,y1,x2,y2=line[0]


        LINE_OBJECTS.append(

            {


            "type":
            "LINE",


            "start":

            [

            int(x1),

            int(y1)

            ],


            "end":

            [

            int(x2),

            int(y2)

            ]

            }

        )



print(

len(LINE_OBJECTS)

)

In [ ]:
# ======================================================
# Circle Detection
# ======================================================


circles=cv2.HoughCircles(

    edges,

    cv2.HOUGH_GRADIENT,

    dp=1,

    minDist=30,

    param1=100,

    param2=30,

    minRadius=5,

    maxRadius=500

)



CIRCLE_OBJECTS=[]


if circles is not None:


    circles=np.round(

        circles[0]

    ).astype(int)


    for x,y,r in circles:


        CIRCLE_OBJECTS.append(

            {


            "type":
            "CIRCLE",


            "center":

            [

            int(x),

            int(y)

            ],


            "radius":

            int(r)


            }

        )


print(

"Circle:",

len(CIRCLE_OBJECTS)

)

In [ ]:
# ======================================================
# Contour
# ======================================================


contours,_=cv2.findContours(

    edges,

    cv2.RETR_EXTERNAL,

    cv2.CHAIN_APPROX_SIMPLE

)



print(

"Contours:",

len(contours)

)

In [ ]:
# ======================================================
# Polyline
# ======================================================


POLYGON_OBJECTS=[]


for c in contours:


    area=cv2.contourArea(c)


    if area < 100:

        continue



    epsilon = 0.01 * cv2.arcLength(

        c,

        True

    )


    approx=cv2.approxPolyDP(

        c,

        epsilon,

        True

    )


    points=[]


    for p in approx:


        x,y=p[0]


        points.append(

            [

            int(x),

            int(y)

            ]

        )



    POLYGON_OBJECTS.append(

        {


        "type":
        "POLYLINE",


        "points":
        points


        }

    )



print(

"Polyline:",

len(POLYGON_OBJECTS)

)

In [ ]:
# ======================================================
# Geometry Database
# ======================================================


GEOMETRY_OBJECTS=[]


GEOMETRY_OBJECTS.extend(

    LINE_OBJECTS

)


GEOMETRY_OBJECTS.extend(

    CIRCLE_OBJECTS

)


GEOMETRY_OBJECTS.extend(

    POLYGON_OBJECTS

)



print(

"Total:",

len(GEOMETRY_OBJECTS)

)

In [ ]:
# ======================================================
# Preview Vector
# ======================================================


canvas=np.zeros_like(
    edges
)



for obj in GEOMETRY_OBJECTS:


    if obj["type"]=="LINE":


        cv2.line(

            canvas,

            obj["start"],

            obj["end"],

            255,

            2

        )


    elif obj["type"]=="CIRCLE":


        cv2.circle(

            canvas,

            obj["center"],

            obj["radius"],

            255,

            2

        )


    elif obj["type"]=="POLYLINE":


        pts=np.array(

            obj["points"]

        )


        cv2.polylines(

            canvas,

            [

            pts

            ],

            True,

            255,

            2

        )



plt.figure(
figsize=(10,10)
)

plt.imshow(
canvas,
cmap="gray"
)

plt.title(
"Vector Preview"
)

plt.axis("off")

plt.show()

In [ ]:
# ======================================================
# Save Geometry
# ======================================================


GEOMETRY_FILE = (

ROOT /

"temp" /

"geometry_objects.json"

)



with open(

    GEOMETRY_FILE,

    "w",

    encoding="utf-8"

) as f:


    json.dump(

        GEOMETRY_OBJECTS,

        f,

        indent=4

    )



print(

"Saved:",

GEOMETRY_FILE

)

In [ ]:
# ======================================================
# Update State
# ======================================================


PROJECT_STATE.update({

    "geometry_objects":

    str(GEOMETRY_FILE),


    "geometry_count":

    len(GEOMETRY_OBJECTS)

})



with open(

STATE_FILE,

"w",

encoding="utf-8"

) as f:


    json.dump(

        PROJECT_STATE,

        f,

        indent=4

    )



print(
"Geometry State Updated"
)